# Notebook Parameters

In [ ]:
main_dir = '/Users/davide/Developer/ml-tropical-cyclones-detection-release'

# path to dataset directory (if CMIP6 data must be in the proper grid format)
# dataset_dir = f'{main_dir}/data/projections/nicam16-9s/'
dataset_dir = f'{main_dir}/data/datasets/north_pacific/'

# path to model that we want to use in inference
selected_model = '04_vgg_v3_vo_850'
model_dir = f'{main_dir}/backup/{selected_model}'

# path to IBTrACS file to match ML model detections
ibtracs_src = f'{main_dir}/data/ibtracs/filtered/ibtracs_main-tracks_6h_1980-2021_TS-NR-ET-MX-SS-DS.csv'

# year to test
year = 1993

# device used for the inference
device = 'mps'

# define lat and lon ranges
lat_range = (0,70)
lon_range = (100,320)

## Inference Workflow

In [ ]:
import pandas as pd
import torch
import sys
import os

import warnings
warnings.filterwarnings('ignore')

sys.path.append('../resources/library')
from tropical_cyclone.inference import SingleModelInference, get_observations, get_observed_tracks
from tropical_cyclone.visualize import plot_detections, plot_tracks, plot_pod_and_far, plot_track_durations
from tropical_cyclone.cyclone import compute_pod_and_far
from tropical_cyclone.models import *
import dynamicopy

Load ML model and historical data and run inferenence for detecting TC centers

In [ ]:
inference = SingleModelInference(model_dir=model_dir, device=device)
ds, dates = inference.load_dataset(dataset_dir=dataset_dir, year=year)
detections = inference.predict(ds, patch_size=40)

In [ ]:
observations = get_observations(ibtracs_src=ibtracs_src, dates=dates, lat_range=lat_range, lon_range=lon_range)

# Apply Tracking Algorithm

Run tracking algorithm on detected TC centers

In [ ]:
det_tracks = inference.tracking(detections, max_distance=400.0, min_track_count=12)
obs_tracks = get_observed_tracks(observations)

## Detections

Plot detected TC centers

In [ ]:
plot_detections(detections, observations, lat_range, lon_range)

## Tracks

Plot detected TC tracks

In [ ]:
plot_tracks(det_tracks, obs_tracks, lat_range, lon_range)

Computer POD and FAR of detected TC tracks with respect to observations and other trackers

In [ ]:
obs_tracks = obs_tracks.rename(columns={'ISO_TIME':'time','LAT':'lat','LON':'lon','TRACK_ID':'track_id'})[['time','lat','lon','track_id']]
det_tracks = det_tracks.rename(columns={'ISO_TIME':'time', 'LAT':'lat', 'LON':'lon', 'WS':'ws', 'TRACK_ID':'track_id', 'HAVERSINE':'haversine'})

# convert longitudes to range [0, 360] format
obs_tracks['lon'] = (obs_tracks['lon'] + 180) % 360 - 180
det_tracks['lon'] = (det_tracks['lon'] + 180) % 360 - 180

mathces, results = compute_pod_and_far(dynamicopy, det_tracks, selected_model, obs_tracks, 300, print_results=False)
results['pod'] = results['pod'] * 100
results['far'] = results['far'] * 100

plot_pod_and_far(results, '', None)

Plot track durations

In [ ]:
plot_track_durations(selected_model, det_tracks, obs_tracks)